In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class UNet3DDecoder(nn.Module):
    def __init__(self, in_channels=512, out_channels=1, base_channels=32):
        super(UNet3DDecoder, self).__init__()
        
        # Encoder path - reverse order of 3D UNet
        self.up1 = self._up_block(in_channels, base_channels * 4)
        self.up2 = self._up_block(base_channels * 4, base_channels * 2)
        self.up3 = self._up_block(base_channels * 2, base_channels)
        
        # Final conv to output 3D volume (e.g., 64x64x64 or your CT size)
        self.final_conv = nn.Conv3d(base_channels, out_channels, kernel_size=1)

    def _up_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.ConvTranspose3d(in_ch, out_ch, kernel_size=2, stride=2),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        x = self.up1(x)
        x = self.up2(x)
        x = self.up3(x)
        out = self.final_conv(x)
        return out


In [ ]:
class KneeReconstructionModel(nn.Module):
    def __init__(self, encoder_la, encoder_pa, cross_attention, decoder_3d):
        super().__init__()
        self.encoder_la = encoder_la
        self.encoder_pa = encoder_pa
        self.cross_attention = cross_attention
        self.decoder = decoder_3d

    def forward(self, x_la, x_pa):
        feat_la = self.encoder_la(x_la)  # shape: (B, C, H, W)
        feat_pa = self.encoder_pa(x_pa)
        
        # Assume both features are reshaped or expanded to match 3D input
        fused = self.cross_attention(feat_la, feat_pa)  # Custom attention fusion
        fused = fused.unsqueeze(2)  # Expand to 5D if needed, e.g., (B, C, 1, H, W)
        out = self.decoder(fused)
        return out


In [ ]:
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import numpy as np
import os

class KneeXray3DDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.patients = os.listdir(root_dir)
        self.transform = transform

    def __len__(self):
        return len(self.patients)

    def __getitem__(self, idx):
        pid = self.patients[idx]
        img_la = Image.open(os.path.join(self.root_dir, pid, 'DRR/left/la.png')).convert('RGB')
        img_pa = Image.open(os.path.join(self.root_dir, pid, 'DRR/left/pa.png')).convert('RGB')

        if self.transform:
            img_la = self.transform(img_la)
            img_pa = self.transform(img_pa)

        target_path = os.path.join(self.root_dir, pid, 'target.npz')
        target_volume = np.load(target_path)['volume']  # shape: (D, H, W)
        target_volume = torch.tensor(target_volume, dtype=torch.float32).unsqueeze(0)  # (1, D, H, W)

        return img_la, img_pa, target_volume


In [ ]:
def train(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for la, pa, target in dataloader:
        la, pa, target = la.to(device), pa.to(device), target.to(device)
        
        optimizer.zero_grad()
        output = model(la, pa)
        
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    
    return total_loss / len(dataloader)


In [ ]:
import torchvision.transforms as T
from torch.utils.data import DataLoader

# Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
transform = T.Compose([T.Resize((224, 224)), T.ToTensor()])

# Load dataset
dataset = KneeXray3DDataset('data/processed', transform=transform)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# Initialize components
encoder_la = ConvNeXtV2FeatureExtractor().to(device)
encoder_pa = ConvNeXtV2FeatureExtractor().to(device)
cross_attention = CrossAttentionFusion().to(device)
decoder = UNet3DDecoder().to(device)

# Full model
model = KneeReconstructionModel(encoder_la, encoder_pa, cross_attention, decoder).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.MSELoss()

# Training loop
for epoch in range(20):
    loss = train(model, dataloader, optimizer, criterion, device)
    print(f"Epoch {epoch+1}, Loss: {loss:.4f}")


In [3]:
import torch
import torch.nn as nn
from monai.networks.nets import UNet as UNet3D
import timm

class BiPlanarTo3DUNet(nn.Module):
    def __init__(self, input_size=(128, 128), output_shape=(64, 64, 64), backbone='convnextv2_tiny'):
        super(BiPlanarTo3DUNet, self).__init__()

        # Load ConvNeXt V2 from timm (remove classifier)
        self.encoder_la = timm.create_model(backbone, pretrained=True, num_classes=0, features_only=False)
        self.encoder_pa = timm.create_model(backbone, pretrained=True, num_classes=0, features_only=False)

        # Output feature size for convnextv2_tiny is 768
        feature_dim = self.encoder_la.num_features * 2  # concatenate LA and PA features

        # Reduce to 64 channels + prepare spatial structure
        self.reduce = nn.Sequential(
            nn.Unflatten(1, (feature_dim, 1, 1)),                  # (B, 1536) -> (B, 1536, 1, 1)
            nn.Conv2d(feature_dim, 64, kernel_size=1),             # (B, 64, 1, 1)
            nn.ReLU(),
            nn.Upsample(size=(16, 16), mode='bilinear')            # (B, 64, 16, 16)
        )

        # Convert to 3D volume
        self.to_3d = nn.Sequential(
            nn.ConvTranspose3d(64, 32, kernel_size=4, stride=2, padding=1),  # (B, 32, D=2, H=32, W=32)
            nn.ReLU()
        )

        # 3D U-Net
        self.unet3d = UNet3D(
            spatial_dims=3,
            in_channels=32,
            out_channels=1,
            channels=(32, 64, 128, 256),
            strides=(2, 2, 2),
            num_res_units=2,
        )

    def forward(self, x_la, x_pa):
        # Feature embedding (B, 768)
        f_la = self.encoder_la(x_la)  # (B, 768)
        f_pa = self.encoder_pa(x_pa)  # (B, 768)

        # Concatenate features
        fused = torch.cat((f_la, f_pa), dim=1)  # (B, 1536)

        # Reshape and reduce
        reduced = self.reduce(fused)  # (B, 64, 16, 16)

        # Prepare for 3D: (B, 64, 16, 16) -> (B, 64, 1, 16, 16)
        vol_3d = reduced.unsqueeze(2)  # D=1

        # Expand to 3D
        vol_3d = self.to_3d(vol_3d)  # (B, 32, D, H, W)

        # 3D U-Net
        out = self.unet3d(vol_3d)  # (B, 1, D, H, W)
        return out


ImportError: cannot import name 'get_ctx' from 'torch.library' (/opt/anaconda3/envs/3d-recon-ai/lib/python3.11/site-packages/torch/library.py)

In [ ]:
model = BiPlanarTo3DUNet()
x_la = torch.randn(1, 3, 128, 128)  # LA view
x_pa = torch.randn(1, 3, 128, 128)  # PA view

output = model(x_la, x_pa)
print(output.shape)  # (1, 1, 64, 64, 64)


In [19]:
import numpy as np
import open3d as o3d

# Load the npz file
data = np.load('../augmented_bone_data/002_aug0/model_separated/right/bone_3.npz')

# List all arrays inside npz
print("Arrays in npz:", list(data.keys()))

# Replace 'array_name' with the actual array name you want to visualize
volume = data['voxel']
assert volume.ndim == 3
print("Target shape:", volume.shape)

# Threshold to extract points
threshold = volume.mean()  # or choose a fixed value
voxels = np.argwhere(volume > threshold)
points = voxels.astype(float)

# Create Open3D point cloud
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)

# Visualize
o3d.visualization.draw_geometries([pcd])


Arrays in npz: ['voxel']
Target shape: (128, 128, 128)


In [13]:
import numpy as np

volume = np.load('../output/knee_voxel_3.npy')
print("Original shape:", volume.shape)

# Remove batch dimension if present
if volume.shape[0] == 1:
    volume = np.squeeze(volume, axis=0)

assert volume.ndim == 3
print("Squeezed shape:", volume.shape)

# Continue as usual
threshold = volume.mean()
voxels = np.argwhere(volume > threshold)
points = voxels.astype(float)

import open3d as o3d
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)

o3d.visualization.draw_geometries([pcd])


Original shape: (128, 128, 128)
Squeezed shape: (128, 128, 128)
